[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobMulla/torchtpu-inference-workshop/blob/main/notebooks/06_end_to_end_showcase_and_domain_adaptation.ipynb)

# End-to-End TorchTPU Showcase: Multi-Model Domain Adaptation, Pallas Kernel Injection & Prefix-Cached Serving

This master notebook brings together the full **PyTorch on Cloud TPU (`torch_tpu`)** stack, **TPU Agent Suite (`tpu-agent-suite`)**, and **Colab MCP (`colab-mcp`)** into a single end-to-end engineering narrative:

### The Enterprise Story: Building a Low-Latency Compliance & Code Auditing Platform on Cloud TPU
Modern enterprise engineering teams face three compounding challenges when deploying specialized AI guardrails:
1. **Off-the-shelf base models miss domain-specific rules** (e.g., internal PCI-DSS v4.0 payment tokenization rules, ISO-20022 wire idempotency, or deprecated `torch_xla` patterns).
2. **Naive PyTorch eager execution underutilizes TPU Matrix Multiply Units (MXUs)** due to host-device dispatch overhead and HBM round-trips on normalization and gated-MLP ops.
3. **Multi-pass repository & transaction auditing re-computes shared system policies and repo AST headers**, inflating Time-to-First-Token (TTFT).

### The 4-Stage Developer Playbook Demonstrated Here
| Stage | What We Build | Core `torch_tpu` Primitive | Key Takeaway |
| :--- | :--- | :--- | :--- |
| **1. Interactive Bridge** | Connect Colab UI + Coding Agent (`colab-mcp`) to a Cloud TPU (`http://localhost:8888/?token=tpucolab`) | `torch.device("tpu")` + `TPU_VISIBLE_CHIPS` | Zero `torch_xla` boilerplate; native PyTorch device semantics |
| **2. Multi-Model Domain LoRA** | Adapt **Model A (`Qwen/Qwen3-0.6B`)** for SecOps/TPU Migration and **Model B (`Qwen/Qwen2.5-0.5B-Instruct`)** for FinTech PCI-DSS Compliance | Static `bfloat16` shapes + `peft.get_peft_model` + `merge_and_unload()` | Zero XLA recompilation; rapid domain perplexity convergence |
| **3. XProf & Pallas Kernel Patching** | Profile with `ProfilerActivity.TPU`, fuse `RMSNorm` + `SwiGLU` in Pallas, and hot-patch `RMSNorm` and `SwiGLU MLP` submodules inside the adapted Hugging Face model | `torch.tpu.pallas.jax_op` (`pallas::hf_rmsnorm`, `pallas::hf_swiglu`) | Single-program TPU execution & in-VMEM reduction with verified logit parity |
| **4. Prefix-Cached Serving & Burst** | Serve the merged checkpoint on `sglang-torchtpu` with RadixAttention prefix caching and 16-way continuous batching | `--device tpu --enable-torch-compile --enable-cache-report` | Shared policy/repo prefix stays hot in HBM KV cache across audit passes |

**Runtime:** in Colab, *Runtime → Change runtime type → TPU* (or *Connect to a local runtime* → `http://localhost:8888/?token=tpucolab`).

## 1. Environment & Single-Chip TPU Setup

Install `torch_tpu`, `peft`, `transformers`, `jax`, and `matplotlib`, and verify that `torch.device("tpu")` is active.

In [ ]:
import importlib.util, importlib.metadata as md, os

if importlib.util.find_spec("google.colab"):
    for k, v in {
        "TPU_SKIP_MDS_QUERY": "1",
        "TPU_CHIPS_PER_HOST_BOUNDS": "1,1,1",
        "TPU_HOST_BOUNDS": "1,1,1",
        "TPU_WORKER_ID": "0",
        "TPU_WORKER_HOSTNAMES": "localhost",
    }.items():
        os.environ.setdefault(k, v)

if importlib.util.find_spec("torch_tpu") is None:
    %pip install -q torch_tpu==0.2.0 --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q peft transformers accelerate openai requests matplotlib uv

for pkg in ["torch", "torch_tpu", "libtpu", "jax", "transformers", "peft"]:
    print(f"{pkg:14s} {md.version(pkg)}")

## 2. Multi-Model Domain Adaptation + In-Model Pallas Kernel Patching

In this stage we:
1. Fine-tune **`Qwen/Qwen2.5-0.5B-Instruct`** on **FinTech PCI-DSS v4.0 & ISO-20022 Payment Compliance** rules using static `bfloat16` tensors on `torch.device("tpu")`, and compare its convergence and perplexity reduction against **`Qwen/Qwen3-0.6B`** adapted on **SecOps & PyTorch-TPU Migration** rules.
2. Register two custom **Pallas TPU kernels** (`pallas::hf_rmsnorm` and `pallas::hf_swiglu`) via `torch.tpu.pallas.jax_op`.
3. **Hot-patch all 73 `RMSNorm` and `SwiGLU MLP` submodules** inside the adapted Hugging Face model and measure both isolated 24-layer `RMSNorm` speedup and full-model forward latency + logit parity.

In [ ]:
%%writefile domain_adapt_and_pallas_patch.py
import glob, json, math, os, time, warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("TPU_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TPU_STDERR_LOG_LEVEL", "2")
os.environ.setdefault("TORCH_CPP_LOG_LEVEL", "ERROR")
os.environ.pop("TPU_VISIBLE_DEVICES", None)
for k, v in {
    "TPU_VISIBLE_CHIPS": "0",
    "TPU_SKIP_MDS_QUERY": "1",
    "TPU_CHIPS_PER_HOST_BOUNDS": "1,1,1",
    "TPU_HOST_BOUNDS": "1,1,1",
    "TPU_WORKER_ID": "0",
    "TPU_WORKER_HOSTNAMES": "localhost",
}.items():
    os.environ[k] = v

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_tpu
from torch_tpu import _loader

if not _loader._LOADED:
    _loader._init_device("tpu")

_init_tensor = torch.ones(1, device="tpu")
torch.accelerator.synchronize()

import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

def _rmsnorm_pallas_kernel(x_ref, w_ref, out_ref):
    x = x_ref[...].astype(jnp.float32)
    sq_mean = jnp.mean(x * x, axis=-1, keepdims=True)
    normed = (x * jax.lax.rsqrt(sq_mean + 1e-6)).astype(out_ref.dtype)
    out_ref[...] = normed * w_ref[...]

def pallas_rmsnorm_jax(x: jax.Array, w: jax.Array) -> jax.Array:
    rows_x = x.reshape(-1, x.shape[-1])
    n, d = rows_x.shape
    tile = 128
    out = pl.pallas_call(
        _rmsnorm_pallas_kernel,
        grid=(n // tile,),
        in_specs=[
            pl.BlockSpec((tile, d), lambda i: (i, 0)),
            pl.BlockSpec((1, d), lambda i: (0, 0)),
        ],
        out_specs=pl.BlockSpec((tile, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(rows_x.shape, rows_x.dtype),
    )(rows_x, w.reshape(1, d))
    return out.reshape(x.shape)

def _swiglu_pallas_kernel(gate_ref, up_ref, out_ref):
    g = gate_ref[...].astype(jnp.float32)
    silu_g = (g * jax.lax.logistic(g)).astype(out_ref.dtype)
    out_ref[...] = silu_g * up_ref[...]

def pallas_swiglu_jax(gate: jax.Array, up: jax.Array) -> jax.Array:
    rows_g = gate.reshape(-1, gate.shape[-1])
    rows_u = up.reshape(-1, up.shape[-1])
    n, d = rows_g.shape
    tile = 128
    out = pl.pallas_call(
        _swiglu_pallas_kernel,
        grid=(n // tile,),
        in_specs=[
            pl.BlockSpec((tile, d), lambda i: (i, 0)),
            pl.BlockSpec((tile, d), lambda i: (i, 0)),
        ],
        out_specs=pl.BlockSpec((tile, d), lambda i: (i, 0)),
        out_shape=jax.ShapeDtypeStruct(rows_g.shape, rows_g.dtype),
    )(rows_g, rows_u)
    return out.reshape(gate.shape)

pallas_hf_rmsnorm = torch.tpu.pallas.jax_op("pallas::hf_rmsnorm", pallas_rmsnorm_jax)
pallas_hf_swiglu = torch.tpu.pallas.jax_op("pallas::hf_swiglu", pallas_swiglu_jax)

class PallasPatchedRMSNorm(nn.Module):
    def __init__(self, orig_norm: nn.Module):
        super().__init__()
        self.weight = orig_norm.weight
        self.variance_epsilon = getattr(orig_norm, "variance_epsilon", 1e-6)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        n_rows = hidden_states.numel() // hidden_states.shape[-1]
        if n_rows >= 128 and n_rows % 128 == 0:
            return pallas_hf_rmsnorm(hidden_states, self.weight)
        dt = hidden_states.dtype
        h = hidden_states.to(torch.float32)
        var = h.pow(2).mean(-1, keepdim=True)
        return self.weight * (h * torch.rsqrt(var + self.variance_epsilon)).to(dt)

class PallasPatchedMLP(nn.Module):
    def __init__(self, orig_mlp: nn.Module):
        super().__init__()
        self.gate_proj = orig_mlp.gate_proj
        self.up_proj = orig_mlp.up_proj
        self.down_proj = orig_mlp.down_proj

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        n_rows = gate.numel() // gate.shape[-1]
        if n_rows >= 128 and n_rows % 128 == 0:
            act = pallas_hf_swiglu(gate, up)
        else:
            act = F.silu(gate) * up
        return self.down_proj(act)

def patch_hf_model_with_pallas(model: nn.Module) -> int:
    patched_count = 0
    inner = model.model if hasattr(model, "model") else model
    if hasattr(inner, "layers"):
        for layer in inner.layers:
            layer.input_layernorm = PallasPatchedRMSNorm(layer.input_layernorm)
            layer.post_attention_layernorm = PallasPatchedRMSNorm(layer.post_attention_layernorm)
            layer.mlp = PallasPatchedMLP(layer.mlp)
            patched_count += 3
        if hasattr(inner, "norm"):
            inner.norm = PallasPatchedRMSNorm(inner.norm)
            patched_count += 1
    return patched_count

FINTECH_CORPUS = [
    {
        "instruction": "Audit this payment microservice snippet for PCI-DSS v4.0 and TPU latency compliance:\n`logger.info(f'Charging PAN={card_number} CVV={cvv} amount={amt}')`",
        "response": "SEVERITY: CRITICAL | RULE: PCI-DSS-3.3.1 | VIOLATION: Full Primary Account Number (PAN) and CVV logged in cleartext. REMEDIATION: Mask PAN to first 6 / last 4 digits (`tokenize_pan(card_number)`) and never log SAD/CVV.",
    },
    {
        "instruction": "Audit this ISO-20022 settlement batch processor for TPU execution hazards:\n`for tx in transactions:\n    if tx.risk_score.item() > 0.85: flag(tx)`",
        "response": "SEVERITY: HIGH | RULE: TPU-SYNC-02 | VIOLATION: Per-transaction `.item()` triggers host-device scalar transfer inside payment settlement loop. REMEDIATION: Vectorize threshold mask `high_risk = (scores > 0.85)` on TPU.",
    },
    {
        "instruction": "Audit this wire-transfer idempotency handler for financial compliance:\n`transfer_id = f'{sender_id}_{int(time.time())}'`",
        "response": "SEVERITY: HIGH | RULE: FIN-IDEMP-01 | VIOLATION: Second-granularity timestamp collision risk in wire transfer idempotency key. REMEDIATION: Use client-supplied UUIDv7 + deterministic payload HMAC-SHA256.",
    },
    {
        "instruction": "Audit this fraud-scoring inference batcher on Cloud TPU:\n`batch = pad_to_dynamic_max(requests); scores = fraud_model(batch.to('tpu'))`",
        "response": "SEVERITY: HIGH | RULE: TPU-SHAPE-01 | VIOLATION: Dynamic batch length causes XLA recompilation spikes exceeding 50ms payment SLA. REMEDIATION: Bucket requests into static shapes `(16, 128)` or `(64, 256)`.",
    },
    {
        "instruction": "Audit this clean PCI-compliant tokenized TPU fraud scorer:\n`pan_tok = vault.tokenize(req.pan); x = static_pad(features, 128).to(device); scores = compiled_scorer(x)`",
        "response": "SEVERITY: PASS | RULE: NONE | VIOLATION: None. PAN is vaulted prior to inference and features use static bucketed TPU shapes.",
    },
    {
        "instruction": "Audit this cryptographic key rotation helper in checkout service:\n`cipher = AES.new(b'0123456789abcdef', AES.MODE_ECB)`",
        "response": "SEVERITY: CRITICAL | RULE: PCI-DSS-3.5.1 | VIOLATION: Hardcoded 128-bit key and insecure AES-ECB mode leak block patterns. REMEDIATION: Fetch DEK from Cloud KMS and use AES-256-GCM with unique 96-bit nonce.",
    },
]

def build_static_batch(tokenizer, examples, seq_len=128, device="tpu"):
    ids_l, mask_l, lbl_l = [], [], []
    for ex in examples:
        prompt = f"### Instruction:\n{ex['instruction']}\n\n### Audit Verdict:\n"
        full = prompt + ex["response"] + tokenizer.eos_token
        p_ids = tokenizer.encode(prompt, add_special_tokens=False)
        f_ids = tokenizer.encode(full, add_special_tokens=False, truncation=True, max_length=seq_len)
        pad = seq_len - len(f_ids)
        ids_l.append(f_ids + [tokenizer.pad_token_id] * pad)
        mask_l.append([1] * len(f_ids) + [0] * pad)
        lbl_l.append(([-100] * len(p_ids) + f_ids[len(p_ids):] + [-100] * pad)[:seq_len])
    return (
        torch.tensor(ids_l, dtype=torch.long, device=device),
        torch.tensor(mask_l, dtype=torch.long, device=device),
        torch.tensor(lbl_l, dtype=torch.long, device=device),
    )

device = torch.device("tpu")
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.bfloat16).to(device)
peft_model = get_peft_model(
    base_model,
    LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias="none", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], task_type="CAUSAL_LM"),
).to(device)

input_ids, attention_mask, labels = build_static_batch(tokenizer, FINTECH_CORPUS, seq_len=128, device=device)
with torch.no_grad():
    pre_loss = float(peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss.cpu().item())

optimizer = torch.optim.AdamW(peft_model.parameters(), lr=5e-4, weight_decay=0.01)
peft_model.train()
for step in range(1, 31):
    t0 = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)
    loss = peft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
    loss.backward()
    optimizer.step()
    torch.accelerator.synchronize()
    if step in (1, 10, 20, 30):
        print(f"Step {step:02d}/30 | Loss={loss.item():.4f} | {(time.perf_counter()-t0)*1000:.1f} ms")

merged = peft_model.merge_and_unload().eval()
with torch.no_grad():
    ref_logits = merged(input_ids=input_ids).logits.clone()
    torch.accelerator.synchronize()
    eager_t = []
    for _ in range(15):
        t0 = time.perf_counter()
        _ = merged(input_ids=input_ids).logits
        torch.accelerator.synchronize()
        eager_t.append((time.perf_counter() - t0) * 1000.0)
    patched_n = patch_hf_model_with_pallas(merged)
    pal_logits = merged(input_ids=input_ids).logits.clone()
    torch.accelerator.synchronize()
    pal_t = []
    for _ in range(15):
        t0 = time.perf_counter()
        _ = merged(input_ids=input_ids).logits
        torch.accelerator.synchronize()
        pal_t.append((time.perf_counter() - t0) * 1000.0)

e_ms = sorted(eager_t)[len(eager_t)//2]
p_ms = sorted(pal_t)[len(pal_t)//2]
agree = float((pal_logits.argmax(-1) == ref_logits.argmax(-1)).float().mean().cpu().item()) * 100.0
print(f"Patched {patched_n} submodules | Eager={e_ms:.2f} ms -> Pallas={p_ms:.2f} ms ({e_ms/p_ms:.2f}x) | Top-1 Agreement={agree:.2f}%")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "domain_adapt_and_pallas_patch.py"], check=True)

## 3. Run the 4-Mode AutoKernel Lab & SpecGuard SGLang Pipeline

You can also run the standalone `autokernel_lab.py` (4-way comparison of `Eager`, `DeferAndFuse`, `torch.compile`, and `Pallas Fused Kernels` with XProf trace analysis) and `SpecGuard TPU` (`stage1_lora_finetune.py` + `stage2_serve_and_audit.py`) directly from this runtime.

In [ ]:
import os, IPython.display as disp

for img_path in [
    "../demos/07_domain_adaptation_showcase/domain_adaptation_showcase.png",
    "../demos/06_autokernel_app/autokernel_xprof_waterfall.png",
    "../demos/05_specguard_app/specguard_dashboard.png",
]:
    if os.path.exists(img_path):
        disp.display(disp.Image(filename=img_path))